# Knorm Prompt-Level Evaluation

This notebook uses benchmark `predictions.csv` files that already exist in Drive. It is built to answer one question:

- Which specific prompts was `knorm` best at?
- Which specific prompts was `knorm` worst at?

The workflow is:

1. Discover existing benchmark runs.
2. Select one matched slice (`knorm` vs `no_compression`, optionally another reference).
3. Load and align the same examples across methods.
4. Compute prompt-level wins, regressions, and efficiency deltas.
5. Show the strongest and weakest prompt families and the exact example rows behind them.


In [ ]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_RESULTS_ROOT = Path('/content/drive/MyDrive/KVCompass/benchmark_eval')
except ImportError:
    DEFAULT_RESULTS_ROOT = Path('results/benchmark_eval')

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 100)

# Change these to the exact slice you want to analyze.
RESULTS_ROOT = DEFAULT_RESULTS_ROOT
DATASET_FILTER = 'ruler'          # Example: 'ruler'
DATA_DIR_FILTER = 4096            # Example: 4096, 8192, 16384
MODEL_FILTER = None               # Example: 'Qwen/Qwen2.5-1.5B-Instruct'
BUDGET_FILTER = None              # Example: 0.5
FRACTION_FILTER = None            # Example: 0.02
SEED_FILTER = None                # Example: 42

KNORM_METHOD = 'knorm'
BASELINE_METHOD = 'no_compression'
REFERENCE_METHOD = None           # Example: 'expected_attention'

TOP_N_BEST = 25
TOP_N_WORST = 25

print('Results root:', RESULTS_ROOT)


In [ ]:
def _read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text()) if path.exists() else {}


def _read_json(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}


def _collect_metric_values(node):
    values = []
    if isinstance(node, dict):
        for value in node.values():
            values.extend(_collect_metric_values(value))
    elif isinstance(node, list):
        for value in node:
            values.extend(_collect_metric_values(value))
    elif isinstance(node, (int, float)):
        values.append(float(node))
    return values


run_rows = []
for run_dir in sorted(RESULTS_ROOT.iterdir() if RESULTS_ROOT.exists() else []):
    if not run_dir.is_dir():
        continue
    predictions_path = run_dir / 'predictions.csv'
    config_path = run_dir / 'config.yaml'
    metrics_path = run_dir / 'metrics.json'
    run_stats_path = run_dir / 'run_stats.json'
    if not predictions_path.exists() or not config_path.exists():
        continue

    config = _read_yaml(config_path)
    metrics = _read_json(metrics_path)
    stats = _read_json(run_stats_path)
    metric_values = _collect_metric_values(metrics)

    run_rows.append({
        'run_dir': str(run_dir),
        'run_name': run_dir.name,
        'dataset': config.get('dataset'),
        'data_dir': config.get('data_dir'),
        'model': config.get('model'),
        'method': config.get('method'),
        'budget': config.get('budget'),
        'fraction': config.get('fraction'),
        'seed': config.get('seed'),
        'predictions_path': str(predictions_path),
        'config_path': str(config_path),
        'metrics_path': str(metrics_path),
        'run_stats_path': str(run_stats_path),
        'avg_quality_from_metrics': np.mean(metric_values) if metric_values else np.nan,
        'avg_latency_seconds': stats.get('avg_latency_seconds'),
        'avg_throughput_tokens_per_second': stats.get('avg_throughput_tokens_per_second'),
        'peak_gpu_memory_mb': stats.get('peak_gpu_memory_mb'),
        'examples': stats.get('examples'),
    })

run_catalog = pd.DataFrame(run_rows).sort_values(
    ['dataset', 'data_dir', 'model', 'method', 'budget'],
    na_position='last'
).reset_index(drop=True)

print('Runs discovered:', len(run_catalog))
display(run_catalog)


In [ ]:
selection = run_catalog.copy()
if DATASET_FILTER is not None:
    selection = selection[selection['dataset'] == DATASET_FILTER]
if DATA_DIR_FILTER is not None:
    selection = selection[selection['data_dir'].astype(str) == str(DATA_DIR_FILTER)]
if MODEL_FILTER is not None:
    selection = selection[selection['model'] == MODEL_FILTER]
if BUDGET_FILTER is not None:
    selection = selection[np.isclose(selection['budget'].astype(float), float(BUDGET_FILTER), equal_nan=False)]
if FRACTION_FILTER is not None:
    selection = selection[np.isclose(selection['fraction'].astype(float), float(FRACTION_FILTER), equal_nan=False)]
if SEED_FILTER is not None:
    selection = selection[selection['seed'].astype(str) == str(SEED_FILTER)]

print('Candidate runs after filters:', len(selection))
display(selection)

def _pick_method_row(df: pd.DataFrame, method_name: str) -> pd.Series:
    matches = df[df['method'] == method_name]
    if matches.empty:
        raise ValueError(f'No run found for method={method_name!r} with the current filters.')
    if len(matches) > 1:
        print(f'Multiple runs matched method={method_name!r}; using the first one. Refine filters if needed.')
        display(matches)
    return matches.iloc[0]

knorm_run = _pick_method_row(selection, KNORM_METHOD)
baseline_run = _pick_method_row(selection, BASELINE_METHOD)
reference_run = _pick_method_row(selection, REFERENCE_METHOD) if REFERENCE_METHOD else None

comparison_runs = pd.DataFrame([row for row in [knorm_run, baseline_run, reference_run] if row is not None])
display(comparison_runs[['run_name', 'dataset', 'data_dir', 'model', 'method', 'budget', 'fraction', 'seed', 'examples']])


In [ ]:
def load_predictions(run_row: pd.Series, prefix: str) -> pd.DataFrame:
    df = pd.read_csv(run_row['predictions_path'])
    df = df.copy()
    df['example_index'] = np.arange(len(df))
    rename_map = {
        column: f'{prefix}_{column}'
        for column in df.columns
    }
    df = df.rename(columns=rename_map)
    return df


def parse_answer_like(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []

    import re

    # Handle numpy-style array strings like ['a' 'b' 'c'] before literal_eval,
    # because Python will concatenate adjacent string literals into one element.
    if text.startswith('[') and text.endswith(']') and ',' not in text:
        matches = re.findall(r'''['"]([^'"]+)['"]''', text)
        flattened = [item.strip() for item in matches if item.strip()]
        if flattened:
            return flattened

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    matches = re.findall(r'''['"]([^'"]+)['"]''', text)
    flattened = [item.strip() for item in matches if item.strip()]
    if flattened:
        return flattened
    return [text]


def _normalize_text(text: str) -> str:
    return ' '.join(str(text).lower().split())


def _extract_terms(text: str) -> set[str]:
    import re
    normalized = _normalize_text(text)
    return set(re.findall(r"[a-z0-9][a-z0-9\-']*", normalized))


def ruler_row_score(task, prediction, reference) -> float:
    pred = _normalize_text(prediction)
    refs = [_normalize_text(x) for x in parse_answer_like(reference)]
    refs = [x for x in refs if x]
    if not pred or not refs:
        return 0.0

    task_name = str(task).lower()
    task_category = task_name.split('_')[0]

    if task_category == 'qa':
        matches = [1.0 if ref in pred else 0.0 for ref in refs]
        return max(matches)

    pred_terms = _extract_terms(pred)
    ref_terms = []
    for ref in refs:
        tokens = list(_extract_terms(ref))
        if len(tokens) == 1:
            ref_terms.append(tokens[0])
        else:
            ref_terms.append(ref)

    matches = []
    for ref in ref_terms:
        if ' ' in ref:
            matches.append(1.0 if ref in pred else 0.0)
        else:
            matches.append(1.0 if ref in pred_terms else 0.0)
    return sum(matches) / len(matches)


def generic_row_score(prediction, reference) -> float:
    pred = _normalize_text(prediction)
    refs = [_normalize_text(x) for x in parse_answer_like(reference)]
    refs = [x for x in refs if x]
    if not pred or not refs:
        return 0.0
    return max(1.0 if ref in pred else 0.0 for ref in refs)


def compute_score(dataset_name, task, prediction, reference) -> float:
    if str(dataset_name).lower() == 'ruler':
        return ruler_row_score(task, prediction, reference)
    return generic_row_score(prediction, reference)


knorm_df = load_predictions(knorm_run, 'knorm')
baseline_df = load_predictions(baseline_run, 'baseline')
reference_df = load_predictions(reference_run, 'reference') if reference_run is not None else None

dataset_name = knorm_run['dataset']

preferred_join_keys = [
    'task',
    'question',
    'answer',
    'context',
    'input',
    'needle',
    'needle_depth',
    'difficulty',
    'length',
    'max_new_tokens',
    'answer_prefix',
]

join_keys = []
for key in preferred_join_keys:
    left = f'knorm_{key}'
    right = f'baseline_{key}'
    if left in knorm_df.columns and right in baseline_df.columns:
        join_keys.append((left, right, key))

if not join_keys:
    join_keys = [('knorm_example_index', 'baseline_example_index', 'example_index')]

left_on = [left for left, _, _ in join_keys]
right_on = [right for _, right, _ in join_keys]
analysis_df = knorm_df.merge(
    baseline_df,
    left_on=left_on,
    right_on=right_on,
    how='inner',
    suffixes=('', ''),
)

if reference_df is not None:
    reference_left_on = []
    reference_right_on = []
    for _, _, key in join_keys:
        left_name = key if key in analysis_df.columns else f'knorm_{key}'
        right_name = f'reference_{key}'
        if left_name in analysis_df.columns and right_name in reference_df.columns:
            reference_left_on.append(left_name)
            reference_right_on.append(right_name)
        else:
            reference_left_on = []
            reference_right_on = []
            break
    if reference_left_on:
        analysis_df = analysis_df.merge(
            reference_df,
            left_on=reference_left_on,
            right_on=reference_right_on,
            how='left',
        )

for _, _, canonical_name in join_keys:
    knorm_name = f'knorm_{canonical_name}'
    baseline_name = f'baseline_{canonical_name}'
    if canonical_name not in analysis_df.columns:
        if knorm_name in analysis_df.columns:
            analysis_df[canonical_name] = analysis_df[knorm_name]
        elif baseline_name in analysis_df.columns:
            analysis_df[canonical_name] = analysis_df[baseline_name]

analysis_df['parsed_answer_items'] = analysis_df.apply(
    lambda row: parse_answer_like(row.get('knorm_answer', row.get('answer'))),
    axis=1,
)
analysis_df['knorm_score'] = analysis_df.apply(
    lambda row: compute_score(
        dataset_name,
        row.get('knorm_task', row.get('task')),
        row.get('knorm_predicted_answer'),
        row.get('knorm_answer', row.get('answer')),
    ),
    axis=1,
)
analysis_df['baseline_score'] = analysis_df.apply(
    lambda row: compute_score(
        dataset_name,
        row.get('baseline_task', row.get('task')),
        row.get('baseline_predicted_answer'),
        row.get('baseline_answer', row.get('answer')),
    ),
    axis=1,
)
if 'reference_predicted_answer' in analysis_df.columns:
    analysis_df['reference_score'] = analysis_df.apply(
        lambda row: compute_score(
            dataset_name,
            row.get('reference_task', row.get('task')),
            row.get('reference_predicted_answer'),
            row.get('reference_answer', row.get('answer')),
        ),
        axis=1,
    )

analysis_df['knorm_hit'] = analysis_df['knorm_score'] >= 0.999
analysis_df['baseline_hit'] = analysis_df['baseline_score'] >= 0.999
if 'reference_score' in analysis_df.columns:
    analysis_df['reference_hit'] = analysis_df['reference_score'] >= 0.999

analysis_df['score_delta'] = analysis_df['knorm_score'] - analysis_df['baseline_score']
analysis_df['knorm_only_win'] = analysis_df['score_delta'] > 1e-9
analysis_df['knorm_only_fail'] = analysis_df['score_delta'] < -1e-9
analysis_df['both_hit'] = analysis_df['knorm_hit'] & analysis_df['baseline_hit']
analysis_df['both_miss'] = ~(analysis_df['knorm_hit'] | analysis_df['baseline_hit'])

analysis_df['latency_delta_seconds'] = analysis_df.get('baseline_latency_seconds') - analysis_df.get('knorm_latency_seconds')
analysis_df['throughput_delta'] = analysis_df.get('knorm_throughput_tokens_per_second') - analysis_df.get('baseline_throughput_tokens_per_second')
analysis_df['memory_delta_mb'] = analysis_df.get('baseline_peak_gpu_memory_mb') - analysis_df.get('knorm_peak_gpu_memory_mb')
analysis_df['output_tokens_delta'] = analysis_df.get('baseline_output_tokens') - analysis_df.get('knorm_output_tokens')

task_source = analysis_df.get('task')
if task_source is None:
    if 'knorm_task' in analysis_df.columns:
        task_source = analysis_df['knorm_task']
    elif 'baseline_task' in analysis_df.columns:
        task_source = analysis_df['baseline_task']
    else:
        task_source = pd.Series(['unknown'] * len(analysis_df))
analysis_df['task_name'] = task_source.astype(str)

print('Rows aligned across methods:', len(analysis_df))
print('Join keys used:', [name for _, _, name in join_keys])
print('Score summary:')
display(analysis_df[['knorm_score', 'baseline_score', 'score_delta']].agg(['mean', 'median', 'min', 'max']))
print('Hit summary:')
display(analysis_df[['knorm_hit', 'baseline_hit', 'knorm_only_win', 'knorm_only_fail']].agg(['mean', 'sum']))
print('Debug preview:')
display(analysis_df[[
    'task_name',
    'parsed_answer_items',
    'knorm_predicted_answer',
    'baseline_predicted_answer',
    'knorm_score',
    'baseline_score',
    'score_delta',
]].head(10))




In [ ]:
def map_task_category(name: str) -> str:
    lowered = str(name).lower()
    if 'needle' in lowered or 'niah' in lowered or 'passkey' in lowered:
        return 'Needle / Retrieval'
    if 'multi' in lowered and 'hop' in lowered:
        return 'Multi-Hop Tracing'
    if 'agg' in lowered or 'aggregation' in lowered or 'common words' in lowered:
        return 'Aggregation'
    if 'qa' in lowered or 'question' in lowered:
        return 'Question Answering'
    if 'code' in lowered:
        return 'Code'
    return 'Other'


analysis_df['task_category'] = analysis_df['task_name'].apply(map_task_category)

category_summary = (
    analysis_df.groupby('task_category', dropna=False)
    .agg(
        prompts=('task_category', 'size'),
        knorm_hit_rate=('knorm_hit', 'mean'),
        baseline_hit_rate=('baseline_hit', 'mean'),
        knorm_only_win_rate=('knorm_only_win', 'mean'),
        knorm_only_fail_rate=('knorm_only_fail', 'mean'),
        avg_latency_gain_seconds=('latency_delta_seconds', 'mean'),
        avg_memory_gain_mb=('memory_delta_mb', 'mean'),
    )
    .sort_values(['knorm_only_fail_rate', 'knorm_only_win_rate'], ascending=[False, False])
)

task_summary = (
    analysis_df.groupby(['task_category', 'task_name'], dropna=False)
    .agg(
        prompts=('task_name', 'size'),
        knorm_hit_rate=('knorm_hit', 'mean'),
        baseline_hit_rate=('baseline_hit', 'mean'),
        knorm_only_win_rate=('knorm_only_win', 'mean'),
        knorm_only_fail_rate=('knorm_only_fail', 'mean'),
        avg_latency_gain_seconds=('latency_delta_seconds', 'mean'),
        avg_memory_gain_mb=('memory_delta_mb', 'mean'),
    )
    .sort_values(['knorm_only_fail_rate', 'prompts'], ascending=[False, False])
)

display(category_summary)
display(task_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
category_plot = category_summary.reset_index()
axes[0].bar(category_plot['task_category'], category_plot['knorm_only_fail_rate'], label='knorm-only fail rate')
axes[0].bar(category_plot['task_category'], category_plot['knorm_only_win_rate'], alpha=0.7, label='knorm-only win rate')
axes[0].set_title('Knorm Regressions vs Wins by Category')
axes[0].set_ylabel('Rate')
axes[0].tick_params(axis='x', rotation=35)
axes[0].legend()

axes[1].scatter(category_plot['avg_latency_gain_seconds'], category_plot['knorm_hit_rate'] - category_plot['baseline_hit_rate'])
for _, row in category_plot.iterrows():
    axes[1].annotate(row['task_category'], (row['avg_latency_gain_seconds'], row['knorm_hit_rate'] - row['baseline_hit_rate']))
axes[1].set_title('Latency Gain vs Hit-Rate Change')
axes[1].set_xlabel('Baseline latency - knorm latency (seconds)')
axes[1].set_ylabel('Knorm hit rate - baseline hit rate')
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
display_columns = [
    column for column in [
        'task_category',
        'task_name',
        'question',
        'knorm_question',
        'answer',
        'knorm_answer',
        'knorm_predicted_answer',
        'baseline_predicted_answer',
        'reference_predicted_answer',
        'knorm_input_tokens',
        'baseline_input_tokens',
        'knorm_output_tokens',
        'baseline_output_tokens',
        'knorm_score',
        'baseline_score',
        'score_delta',
        'latency_delta_seconds',
        'memory_delta_mb',
        'throughput_delta',
        'knorm_hit',
        'baseline_hit',
        'knorm_only_win',
        'knorm_only_fail',
    ] if column in analysis_df.columns
]

best_prompts = analysis_df.copy()
best_prompts = best_prompts.sort_values(
    ['score_delta', 'latency_delta_seconds', 'memory_delta_mb'],
    ascending=[False, False, False],
)

worst_prompts = analysis_df.copy()
worst_prompts = worst_prompts.sort_values(
    ['score_delta', 'latency_delta_seconds', 'memory_delta_mb'],
    ascending=[True, False, False],
)

print('Best prompts for knorm (highest knorm_score - baseline_score):')
display(best_prompts[display_columns].head(TOP_N_BEST))

print('Worst prompts for knorm (lowest knorm_score - baseline_score):')
display(worst_prompts[display_columns].head(TOP_N_WORST))

best_output_path = RESULTS_ROOT / 'knorm_best_prompts.csv'
worst_output_path = RESULTS_ROOT / 'knorm_worst_prompts.csv'
best_prompts.to_csv(best_output_path, index=False)
worst_prompts.to_csv(worst_output_path, index=False)
print('Saved:', best_output_path)
print('Saved:', worst_output_path)



In [ ]:
length_source = None
for candidate in ['knorm_input_tokens', 'baseline_input_tokens', 'length']:
    if candidate in analysis_df.columns:
        length_source = candidate
        break

if length_source is not None:
    bins = [-np.inf, 4096, 8192, 16384, np.inf]
    labels = ['<=4K', '4K-8K', '8K-16K', '>16K']
    analysis_df['length_bucket'] = pd.cut(pd.to_numeric(analysis_df[length_source], errors='coerce'), bins=bins, labels=labels)

    length_summary = (
        analysis_df.groupby('length_bucket', dropna=False)
        .agg(
            prompts=('length_bucket', 'size'),
            knorm_hit_rate=('knorm_hit', 'mean'),
            baseline_hit_rate=('baseline_hit', 'mean'),
            knorm_only_fail_rate=('knorm_only_fail', 'mean'),
            knorm_only_win_rate=('knorm_only_win', 'mean'),
        )
    )
    display(length_summary)

example_columns = [column for column in ['task_category', 'task_name', 'question', 'knorm_question', 'answer', 'knorm_predicted_answer', 'baseline_predicted_answer', 'latency_delta_seconds', 'memory_delta_mb'] if column in analysis_df.columns]

print('Representative regressions to inspect manually:')
for category in worst_prompts['task_category'].dropna().unique()[:4]:
    print(f'\n## {category}')
    display(worst_prompts[worst_prompts['task_category'] == category][example_columns].head(5))
else:
    print('No token-length column was available for bucketing.')


## How To Determine Where Knorm Is Best And Worst

Use the notebook outputs in this order:

1. In the run catalog, filter down to one valid matched slice: same dataset, context length, model, fraction, and seed.
2. In `category_summary` and `task_summary`, look for high `knorm_only_fail_rate` values. Those are the task families where compression is hurting `knorm` most.
3. In `worst_prompts`, inspect rows where `baseline_hit=True` and `knorm_hit=False`. These are the exact prompt-level regressions caused by `knorm`.
4. In `best_prompts`, look for rows where `knorm_hit=True` and `latency_delta_seconds > 0` or `memory_delta_mb > 0`. These are the prompt types where `knorm` preserved quality while buying efficiency.
5. Use the representative regression tables to read several failures manually. Look for recurring patterns such as retrieval vs multi-hop, distractor-heavy tasks, long contexts, or structured prompts.
6. Repeat the same notebook on another context length or another budget. If the same families keep appearing in `worst_prompts`, that is a real `knorm` weakness rather than noise.

A strong conclusion should be phrased in terms of repeated prompt families, not one-off examples. Example: `knorm` is good on localized retrieval prompts with short answers, but degrades on multi-hop tracing where the answer depends on preserving distributed evidence.
